# Timestamp Utility

This notebook provides utility functions for timestamp manipulation, including:
- Fetching the current Unix timestamp in IST (Indian Standard Time)
- Converting Unix timestamps to human-readable formats
- Retrieving the current date
- Performing other date and time calculations

Used as a shared utility imported by ingestion notebooks.

## Import necessary packages

In [ ]:
# Import necessary packages
from datetime import datetime, timezone, timedelta

## Get Unix timestamps in IST

Generates a list of (start, end) Unix timestamp pairs (in milliseconds) for a given date range.

**Supported modes:**
- Both `start_date` and `end_date` provided → returns daily timestamp pairs for the full range
- Only `start_date` provided → returns a single pair from start_date to start_date + 1 day
- Neither provided → defaults to the previous day (UTC midnight in IST)

In [ ]:
def get_unix_timestamps(start_date=None, end_date=None):
    """
    Returns a list of (start_unix_ts, end_unix_ts) tuples in milliseconds (IST).

    Args:
        start_date (str, optional): Start date in 'YYYY-MM-DD HH:MM:SS' or 'YYYY-MM-DD' format.
        end_date (str, optional): End date in 'YYYY-MM-DD HH:MM:SS' or 'YYYY-MM-DD' format.

    Returns:
        list of tuples: [(start_ts_ms, end_ts_ms), ...]
    """
    ist_offset = 19800000  # 5:30 hrs in milliseconds

    def to_ist_datetime(dt):
        return dt - timedelta(milliseconds=ist_offset)

    if start_date and end_date:
        try:
            start_datetime = datetime.strptime(start_date, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            try:
                start_datetime = datetime.strptime(start_date, '%Y-%m-%d')
                start_datetime = start_datetime.replace(hour=0, minute=0, second=0)
            except ValueError:
                print("Invalid start date format. Please provide a valid date in YYYY-MM-DD HH:MM:SS or YYYY-MM-DD format")
                return []

        try:
            end_datetime = datetime.strptime(end_date, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            try:
                end_datetime = datetime.strptime(end_date, '%Y-%m-%d')
                end_datetime = end_datetime.replace(hour=0, minute=0, second=0) + timedelta(days=1)
            except ValueError:
                print("Invalid end date format. Please provide a valid date in YYYY-MM-DD HH:MM:SS or YYYY-MM-DD format")
                return []

        current_datetime = start_datetime
        timestamps = []

        while current_datetime < end_datetime:
            start_unix_timestamp = int(to_ist_datetime(current_datetime).timestamp() * 1000)
            end_of_period = current_datetime + timedelta(days=1)
            end_of_period = end_of_period.replace(hour=0, minute=0, second=0, microsecond=0)

            if end_of_period > end_datetime:
                end_of_period = end_datetime

            end_unix_timestamp = int(to_ist_datetime(end_of_period).timestamp() * 1000)
            timestamps.append((start_unix_timestamp, end_unix_timestamp))
            current_datetime = end_of_period

        return timestamps

    elif start_date and not end_date:
        try:
            start_datetime = datetime.strptime(start_date, '%Y-%m-%d %H:%M:%S')
        except ValueError:
            try:
                start_datetime = datetime.strptime(start_date, '%Y-%m-%d')
                start_datetime = start_datetime.replace(hour=0, minute=0, second=0)
            except ValueError:
                print("Invalid start date format. Please provide a valid date in YYYY-MM-DD HH:MM:SS or YYYY-MM-DD format")
                return []

        end_datetime = start_datetime + timedelta(days=1)
        start_unix_timestamp = int(to_ist_datetime(start_datetime).timestamp() * 1000)
        end_unix_timestamp = int(to_ist_datetime(end_datetime).timestamp() * 1000)
        return [(start_unix_timestamp, end_unix_timestamp)]

    else:
        current_datetime = datetime.combine(datetime.now(timezone.utc).date(), datetime.min.time(), timezone.utc)
        start_unix_timestamp = int(to_ist_datetime(current_datetime - timedelta(days=1)).timestamp() * 1000)
        end_unix_timestamp = int(to_ist_datetime(current_datetime).timestamp() * 1000)
        return [(start_unix_timestamp, end_unix_timestamp)]

## Convert Unix timestamp to IST date

Converts a Unix timestamp (milliseconds) to an IST date string in `YYYY-MM-DD` format.

In [ ]:
def convert_unix_timestamp_to_date(unix_timestamp):
    """
    Converts a Unix timestamp in milliseconds to an IST date string.

    Args:
        unix_timestamp (int): Unix timestamp in milliseconds.

    Returns:
        str: Date string in 'YYYY-MM-DD' format (IST).
    """
    # Convert Unix timestamp (ms) to UTC datetime
    utc_datetime = datetime.utcfromtimestamp(unix_timestamp / 1000)

    # IST is UTC + 5 hours 30 minutes
    ist_offset = timedelta(hours=5, minutes=30)
    ist_datetime = utc_datetime + ist_offset

    # Normalise to midnight
    ist_datetime = ist_datetime.replace(hour=0, minute=0, second=0, microsecond=0)

    # Format as required
    return ist_datetime.strftime('%Y-%m-%d %H:%M:%S')

## Get current date in IST

Returns the current date in IST (Indian Standard Time) as a string in `YYYY-MM-DD HH:MM:SS` format, with time set to `00:00:00`.

Adds 5 hours 30 minutes to UTC to get IST, then normalises to midnight.

In [ ]:
def get_current_date():
    """
    Returns the current date in IST as a formatted string with time set to midnight.

    Returns:
        str: Current IST date string in 'YYYY-MM-DD HH:MM:SS' format (time = 00:00:00).
    """
    # Add 5 hours and 30 minutes to get the date in IST
    date = datetime.now() + timedelta(hours=5, minutes=30)

    # Set time to 00:00:00 and return the formatted date string
    date = date.replace(hour=0, minute=0, second=0, microsecond=0)
    return date.strftime('%Y-%m-%d %H:%M:%S')